# Agent eval — tool-call accuracy, goal accuracy, topic adherence

RAG metrics evaluate one retrieve-and-generate step. Agents chain *many* tool calls before answering, and can fail in places RAG metrics can't see:

| Metric | Catches |
|---|---|
| **Tool-call accuracy** | wrong tool selected, or right tool called with broken args |
| **Agent goal accuracy** | final answer doesn't satisfy the user's stated goal |
| **Topic adherence** | agent wanders mid-trajectory (hallucination across steps) |

We evaluate them on a committed `traces.json` fixture of 5 synthetic traces — one happy path plus four canonical failure modes.

In [ ]:
import os, sys, pathlib, json
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
leaf = pathlib.Path.cwd()
sys.path.insert(0, str(leaf))
from eval import tool_call_accuracy, agent_goal_accuracy, topic_adherence
TRACES = json.loads((leaf / 'traces.json').read_text(encoding='utf-8'))
for t in TRACES:
    print(f"{t['id']:12s}  expects={t['expected_tools']}")

## Trace anatomy

Each trace is a sequence of dicts with one of two roles:

* `tool_call` — `name`, `arguments`, `output_summary`
* `final_answer` — `content`

This schema is intentionally portable: a LangGraph / Pydantic AI / OpenAI Agents SDK trace can be projected onto it with ~10 lines of code.

In [ ]:
happy = TRACES[0]
for step in happy['trace']:
    if step['role'] == 'tool_call':
        print(f"-> {step['name']}({step['arguments']})")
        print(f"   = {step['output_summary']}")
    else:
        print('ANSWER:', step['content'][:120], '...')

## Run all three metrics on every trace

In [ ]:
print(f'{"trace":<12}  {"tool":>6}  {"goal":>6}  {"topic":>6}')
for t in TRACES:
    final = next((s['content'] for s in reversed(t['trace']) if s.get('role') == 'final_answer'), '')
    tca = tool_call_accuracy(t['trace'], t['expected_tools'])
    aga = agent_goal_accuracy(t['user_goal'], final, t['trace'])
    ta = topic_adherence(t['user_goal'], t['trace'])
    print(f'{t["id"]:<12}  {tca:>6.2f}  {aga:>6.2f}  {ta:>6.2f}')

## What each failure mode shows

* `wrong-tool` — `tool_call_accuracy` drops; the other two stay high. This is the *only* metric that catches it.
* `wrong-args` — tool accuracy looks fine because the right tool was called; **goal accuracy** is the metric that catches the bad-args case (no useful answer).
* `off-topic` — tool calls are fine but the final answer talks about the weather. **Topic adherence** drops.
* `missed-goal` — tool accuracy fine, topic adherence fine, but the answer hedges ("somewhere in the paper"). **Goal accuracy** is the metric that catches it.

Three metrics, three failure modes. Each one alone misses two of the four bugs.

## What you can extend

* Plug in a real LangGraph / Pydantic AI / OpenAI Agents SDK trace exporter and convert to this schema.
* Add a 4th metric **step efficiency** — number of tool calls vs. minimum needed.
* Use **paired bootstrap** between two agent configurations to test for statistical significance on a 50–100 trace set.
* Cross-link this with `04-human-in-the-loop/` once that phase ships — HITL gates are the natural mitigation for low goal-accuracy traces.